# Microproyecto 1
- Claudia Bojacá
- Luis Enrique Cabrera

Desarrollar una solución, basada en técnicas de procesamiento del lenguaje natural y machine learning, que permita clasificar automáticamente un texto según los 17 ODS, ofreciendo una forma de presentación de resultados a través de una herramienta de fácil comprensión para el usuario final.

## 1. Importación de librerías requeridas
Se importan las librerías pandas, numpy, nltk y scikit-learn.\
En particular, usaremos las siguientes clases y funciones para realizar el procesamiento:
Sección nueva
- **word_tokenize:** función para separar un texto en una lista de tokens.
- **RegexpTokenizer():** clase para separar un texto en tokens, utilizando expresiones regulares.
- **stopwords:** lista de palabras vacías.
- **PorterStemmer():** clase para realizar stemming mediante el método de Porter.
- **WordCloud():** clase para generar una nube de palabras.
- **CountVectorizer:** clase para obtener una representación de bolsa de palabras.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
#from nltk.tokenize import word_tokenize
#from nltk import RegexpTokenizer
from nltk.corpus import stopwords
#from nltk.stem import PorterStemmer
#from nltk.stem import WordNetLemmatizer
#from wordcloud import WordCloud
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 1. Carga de datos
Se realiza la carga de datos usando la función de Pandas read_csv(), especificando la ruta y el separador del archivo.

In [23]:
columns = ['textos','ODS']
file_id = '1hrSzfkhPNmZwq71LFfoeNKQIxNBseGhH'
ruta = f'https://drive.google.com/uc?export=download&id={file_id}'
data = pd.read_csv(ruta, sep=',', names=columns)
data


,textos,ODS
0,textos,ODS
1,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
2,No dejar clara la naturaleza de estos riesgos ...,6
3,"Como resultado, un mayor y mejorado acceso al ...",13
4,Con el Congreso firmemente en control de la ju...,16
...,...,...
9652,Esto implica que el tiempo de las mujeres en e...,5
9653,"Sin embargo, estas fallas del mercado implican...",3
9654,El hecho de hacerlo y cómo hacerlo dependerá e...,9
9655,"Esto se destacó en el primer estudio de caso, ...",6


## 2. Exploración de los datos


Se evalúan duplicados en la columna "textos"




In [24]:
print(data["textos"].is_unique)

True


Se confirma que no hay duplicados

In [25]:
data.duplicated().sum()

np.int64(0)

Se evalúa si hay datos faltantes

In [26]:
data.isna().sum()

,0
textos,0
ODS,0


En la exploración de datos no se evidenció datos duplicados ni faltantes.

In [27]:
data.shape


(9657, 2)

Se divide el conjunto de datos en entrenamiento y pruebas usando la librería scikit-learn. Se utiliza un split estratificado (stratify=y) para garantizar que en el conjunto test estén representadas todas las clases (17 ODS).

In [28]:
X = data["textos"]
y = data["ODS"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0
)

## 3. Procesamiento de texto
Es necesario ajustar el texto para facilitar el análisis a través de los algoritmos de machine learning. El procesamiento incluye Tokenización, eliminación de palabras vacías y lematización/steamming.
- **Conversión del texto a minúsculas**: Se homogeniza todo el texto para simplificar el procesamiento y el entrenamiento del modelo.
- **Eliminar Palabras vacías**: Se eliminan las palabras irrelevantes para hacer el modelo más efectivo.
- **Tokenizar**: Consiste en la separación del texto en una lista de tokens para facilitar el entrenamiento del modelo.
- **Lematización**: Se ha decidido no aplicar lematización debido a que la lematización requiere recursos adicionales (diccionarios, etiquetado POS, modelos como spaCy), lo que aumenta el tiempo de cómputo y la complejidad del pipeline, mientras que la mejora de desempeño suele ser marginal respecto a una normalización simple (lowercase + stopwords).


In [29]:
nltk_stopwords = stopwords.words("spanish")
tfidVectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=nltk_stopwords,
    analyzer='word',
    ngram_range=(1,1),
    max_features=10000
)

## 4. Reducción de dimensionalidad ##
Se estableció como límite 10.000 como parámetro para la cantidad de palabras (dimensiones). Esta grán cantidad de variables hace costos el uso de estos modelos. Por eso se utiliza una técnica de reducción de dimensionalidad para reducir el costo computacional del modelo sin perder mucha explicabilidad. Se utilizó la función TruncatedSVD debido a que la matríz de palabras es de tipo disperso.

In [30]:
tsvd = TruncatedSVD(n_components=100, random_state=32)

## 5. Modelo de clasificación.
Se utiliza un modelo de regrsión logística para clasificar los textos de acuerdo al ODS al que se refieren

In [31]:
logreg = LogisticRegression(max_iter=500, random_state=32)

## 6. Construcción del pipeline
Se integran los pasos de preprocesamiento, reducción de dimensionalidad y entrenamiento del modelo en un pipeline

In [32]:
steps = [
    ("vectorizer", tfidVectorizer),
    ("tsvd", tsvd),
    ("model", logreg),
]
pipeline = Pipeline(steps)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(max_features=10000,
                                 stop_words=['de', 'la', 'que', 'el', 'en', 'y',
                                             'a', 'los', 'del', 'se', 'las',
                                             'por', 'un', 'para', 'con', 'no',
                                             'una', 'su', 'al', 'lo', 'como',
                                             'más', 'pero', 'sus', 'le', 'ya',
                                             'o', 'este', 'sí', 'porque', ...])),
                ('tsvd', TruncatedSVD(n_components=100, random_state=32)),
                ('model', LogisticRegression(max_iter=500, random_state=32))])

## 7. Selección del mejor modelo por búsqueda de hiperparámetros
Se definió un proceso de selección de hiperparámetros así:
- N-gramas: análisis de solo unigramas (una palabra) o bigramas (una o dos palabras)
- Cantidad de dimensiones: Escenario con 5 mil dimensiones (intermedio) o con 10 mil imensiones (complejo)
- Cantidad de componentes de reducción de dimensionalidad: 50 componentes (modelo muy reducido) o con 100 componentes (modelo más complejo)
- Parámetro de regularización C diversidad de valores de reularización para elegir
- Tipo de regularización: Se selecciona regularización L2 ya que no es necesario un algoritmo que seleccione las variables más relevantes, ya que en la reducción de dimensionalidad ya se ha simplificado el modelo.

In [33]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
param_grid = {
    "vectorizer__ngram_range": [(1,1), (1,2)],
    "vectorizer__max_features": [5000, 10000],
    "tsvd__n_components": [50, 100],
    "model__C": [0.001, 0.1, 1, 10],
    "model__penalty": ["l2"]
}
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=kfold,
    scoring = 'accuracy',
    n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('vectorizer',
                                        TfidfVectorizer(max_features=10000,
                                                        stop_words=['de', 'la',
                                                                    'que', 'el',
                                                                    'en', 'y',
                                                                    'a', 'los',
                                                                    'del', 'se',
                                                                    'las',
                                                                    'por', 'un',
                                                                    'para',
                                                                    'con', 'no',
                                                                    'una', 'su',
                                                                    'al', 'lo',
                                                                    'como',
                                                                    'más',
                                                                    'pero',
                                                                    'sus', 'le',
                                                                    'ya', 'o',
                                                                    'este',
                                                                    'sí',
                                                                    'porque', ...])),
                                       ('tsvd',
                                        TruncatedSVD(n_components=100,
                                                     random_state=32)),
                                       ('model',
                                        LogisticRegression(max_iter=500,
                                                           random_state=32))]),
             n_jobs=-1,
             param_grid={'model__C': [0.001, 0.1, 1, 10],
                         'model__penalty': ['l2'],
                         'tsvd__n_components': [50, 100],
                         'vectorizer__max_features': [5000, 10000],
                         'vectorizer__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy')

## 8. Modelo resultante

In [34]:
print("Mejores parámetros: {}".format(grid.best_params_))
mejor_modelo = grid.best_estimator_

Mejores parámetros: {'model__C': 10, 'model__penalty': 'l2', 'tsvd__n_components': 100, 'vectorizer__max_features': 10000, 'vectorizer__ngram_range': (1, 2)}


## 9. Evaluación del modelo
Se presentan las métricas de evaluación del modelo con el conjunto de prueba, es decir que no fueron utilizados durante el aprendizaje.

In [35]:
y_pred = mejor_modelo.predict(X_test)

In [36]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.90      0.80      0.85        89
          10       0.75      0.73      0.74        66
          11       0.79      0.84      0.81       107
          12       0.85      0.79      0.82        67
          13       0.90      0.85      0.88        87
          14       0.92      0.82      0.87        80
          15       0.95      0.91      0.93        66
          16       0.93      0.97      0.95       228
           2       0.72      0.82      0.77        71
           3       0.89      0.92      0.91       167
           4       0.93      0.95      0.94       214
           5       0.89      0.94      0.92       218
           6       0.90      0.96      0.93       148
           7       0.89      0.88      0.89       158
           8       0.66      0.49      0.56        94
           9       0.74      0.74      0.74        72

    accuracy                           0.87      1932
   macro avg       0.85   

El modelo logra un accuracy de 0.87 en el conjunto de prueba, lo que significa que en el 87% de las veces clasificará de manera correcta al texto deltro del ODS al que correponde. Se observa una presición relativamente alta para todas las categorías con excepción al ODS 8 en donde a solo alcanza en 66%